# kdd_seq_01_markov_mixture: mixture of first-order Markov chains

Each cluster `k` has its own initial-token distribution and transition matrix. EM gives **soft membership** (posteriors), which is what
you want downstream because hard labels were lossy (label lift ~ +0.002 AUC).

Volume control: each learner's counts are rescaled so the effective sequence length is at most `CAP` transitions
(`CAP=None` switches this off). This keeps a 5,000-event learner from dominating, and from being 100 % certain about its cluster.

Reported per K: BIC, held-out likelihood, quality in the shared evaluation space, Cramer's V with dropout, ARI against a volume-only clustering,
then stability (seed / bootstrap / train->test) and downstream AUC with a **random-label null** as a noise yardstick.

In [ ]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.special import logsumexp
from sklearn.cluster import KMeans
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

WORK = "/content/drive/MyDrive/kdd_seq"
sys.path.insert(0, WORK)
import kdd_seq_common as C

# ------------------------------- CONFIG -------------------------------
K_GRID = list(range(2, 9))
N_INIT = 3            # EM restarts per K (best likelihood kept)
CAP = 50              # effective max transitions per learner (None = no cap)
ALPHA = 0.1           # Laplace smoothing
K_FINAL = None        # None -> argmin BIC among K with min_cluster_share >= MIN_SHARE
MIN_SHARE = 0.03
DOWNSTREAM_KS = None  # None -> [K_FINAL]; e.g. [3, 5] to compare several K downstream
CV_SAMPLE, CV_SPLITS, CV_REPEATS = 60000, 3, 3
SEEDS_STAB, N_BOOT = 5, 8
# -----------------------------------------------------------------------
os.makedirs(f"{WORK}/results", exist_ok=True)
A = C.load_artifacts(WORK)
V = A["vocab"]["V"]; NAMES = A["vocab"]["names"]
F_tr, F_te, y_tr, y_te = A["F_train"], A["F_test"], A["y_train"], A["y_test"]
nev_tr = A["base_train"]["n_events"].values
print("train", F_tr.shape, "test", F_te.shape, "| V =", V, "| baseline features:", A["base_train"].shape[1])

## Model
Parameters live in `log_theta` with layout `[init (V) | transitions (V*V)]`, so E-step = `F @ log_theta.T` and M-step = `F.T @ responsibilities`,
both sparse products (no GPU needed). Initialisation: K-Means on the Hellinger transition profile of a sample (volume-free), or random.

In [ ]:
class MarkovMixture:
    def __init__(self, K, V, alpha=0.1, n_iter=100, tol=1e-6, seed=0, cap=50, init="kmeans"):
        self.K, self.V, self.alpha, self.n_iter, self.tol, self.seed, self.cap, self.init = K, V, alpha, n_iter, tol, seed, cap, init

    # ---- helpers
    def _capped(self, F, use_cap=True):
        if not (use_cap and self.cap):
            return F
        n = np.asarray(F[:, self.V:].sum(1)).ravel()
        return (sp.diags(np.minimum(1.0, self.cap / np.maximum(n, 1))) @ F).tocsr()

    def _profile(self, F):
        Tm = F[:, self.V:].tocsr().astype("float64")
        rs = np.asarray(Tm.sum(1)).ravel(); rs[rs == 0] = 1
        Pm = (sp.diags(1 / rs) @ Tm).tocsr(); Pm.data = np.sqrt(Pm.data)
        return Pm

    def _mstep(self, F, R):
        S = np.asarray(F.T.dot(R)).T                                   # (K, D) expected counts
        V, K = self.V, self.K
        init = S[:, :V] + self.alpha
        init /= init.sum(1, keepdims=True)
        tr = S[:, V:].reshape(K, V, V) + self.alpha
        tr /= tr.sum(2, keepdims=True)
        self.log_theta = np.hstack([np.log(init), np.log(tr).reshape(K, V * V)])
        self.log_pi = np.log(np.maximum(R.mean(0), 1e-8))

    def _loglik(self, F):
        return np.asarray(F.dot(self.log_theta.T)) + self.log_pi       # (N, K) joint log-probs

    # ---- fit / inference
    def fit(self, F, use_cap=True):
        rng = np.random.default_rng(self.seed)
        Fc = self._capped(F, use_cap)
        N, K = Fc.shape[0], self.K
        if self.init == "kmeans":
            sub = rng.choice(N, min(30000, N), replace=False)
            km = KMeans(K, n_init=1, random_state=self.seed).fit(self._profile(Fc[sub]))
            lab = km.predict(self._profile(Fc))
            R = np.full((N, K), 0.1 / K); R[np.arange(N), lab] += 0.9
        else:
            R = rng.dirichlet(np.ones(K), size=N)
        self._mstep(Fc, R)
        prev = -np.inf; self.trace_ = []
        for it in range(self.n_iter):
            L = self._loglik(Fc)
            lse = logsumexp(L, axis=1)
            R = np.exp(L - lse[:, None])
            ll = lse.sum(); self.trace_.append(ll)
            self._mstep(Fc, R)
            if abs(ll - prev) < self.tol * abs(ll):
                break
            prev = ll
        self.loglik_ = ll
        n_free = K * (self.V - 1) + K * self.V * (self.V - 1) + (K - 1)
        self.bic_ = -2 * ll + n_free * np.log(N)                      # pseudo-BIC (counts are capped)
        order = np.argsort(-R.sum(0))                                  # cluster 0 = largest
        self.log_theta, self.log_pi = self.log_theta[order], self.log_pi[order]
        return self

    def predict_proba(self, F, use_cap=True):
        L = self._loglik(self._capped(F, use_cap))
        return np.exp(L - logsumexp(L, axis=1, keepdims=True))

    def predict(self, F, use_cap=True):
        return self.predict_proba(F, use_cap).argmax(1)

    def avg_ll(self, F):
        # per-transition average log-likelihood under each cluster (uncapped): "how typical is this learner of cluster k"
        n = np.asarray(F[:, self.V:].sum(1)).ravel()
        return np.asarray(F.dot(self.log_theta.T)) / np.maximum(n, 1)[:, None]

    def heldout_ll_per_transition(self, F):
        L = np.asarray(F.dot(self.log_theta.T)) + self.log_pi
        return float(logsumexp(L, axis=1).sum() / F[:, self.V:].sum())

def fit_best(F, K, n_init=N_INIT, seed=0):
    best = None
    for i in range(n_init):
        m = MarkovMixture(K, V, ALPHA, seed=seed + i, cap=CAP).fit(F)
        if best is None or m.loglik_ > best.loglik_:
            best = m
    return best

## 1. Choose K: model fit, clustering quality, volume check
`silhouette / DB / CH` are measured in the **shared evaluation space** (`eval_*`), so they can be compared with the later methods.
`ari_vs_volume` near 1 would mean the clusters are just activity tiers again.

In [ ]:
rows, models = [], {}
for K in K_GRID:
    t0 = time.time()
    m = fit_best(F_tr, K); models[K] = m
    lab_tr, lab_te = m.predict(F_tr), m.predict(F_te)
    P_tr = m.predict_proba(F_tr)
    r = dict(K=K, bic=m.bic_, heldout_ll_per_trans=m.heldout_ll_per_transition(F_te),
             mean_max_posterior=float(P_tr.max(1).mean()))
    r.update(C.cluster_quality(A["eval_train"], lab_tr))
    r.update(C.volume_agreement(lab_tr, nev_tr))
    r["cramers_v_train"] = C.cramers_v(lab_tr, y_tr); r["cramers_v_test"] = C.cramers_v(lab_te, y_te)
    rows.append(r); print(f"K={K} done in {time.time()-t0:.0f}s | iters={len(m.trace_)} | min share {r['min_cluster_share']:.3f}")
grid = pd.DataFrame(rows).set_index("K")
grid.round(4).to_csv(f"{WORK}/results/markov_grid.csv")
grid.round(4)

In [ ]:
cand = grid[grid["min_cluster_share"] >= MIN_SHARE]
if K_FINAL is None:
    K_FINAL = int((cand if len(cand) else grid)["bic"].idxmin())
print("K_FINAL =", K_FINAL, "(override K_FINAL in the config cell to choose manually)")
m = models[K_FINAL]
lab_tr, lab_te = m.predict(F_tr), m.predict(F_te)
print(C.cluster_profile(lab_tr, y_tr, nev_tr))

## 2. Interpretation: what does each cluster do?
For each cluster: the most **characteristic** transitions (highest lift `P_k(b|a) / P_global(b|a)`, among transitions carrying a non-trivial share of that cluster's mass)
and the most **frequent** ones. Names follow `source_event_category`; `<gap_...>` tokens are pauses.

In [ ]:
def describe(m, F, top=6, min_mass=0.005):
    K, V = m.K, m.V
    Fc = m._capped(F); R = m.predict_proba(F)
    S = np.asarray(Fc.T.dot(R)).T[:, V:].reshape(K, V, V)             # expected transition counts per cluster
    G = S.sum(0) + m.alpha; Pg = G / G.sum(1, keepdims=True)
    out = {}
    for k in range(K):
        Pk = np.exp(m.log_theta[k, V:]).reshape(V, V)
        mass = S[k] / S[k].sum()
        lift = np.where(mass >= min_mass, Pk / Pg, 0)
        li = np.dstack(np.unravel_index(np.argsort(-lift, axis=None)[:top], lift.shape))[0]
        fi = np.dstack(np.unravel_index(np.argsort(-mass, axis=None)[:top], mass.shape))[0]
        out[k] = dict(
            share=float(R[:, k].mean()),
            characteristic=[f"{NAMES[a]} -> {NAMES[b]} (lift {lift[a,b]:.1f})" for a, b in li],
            frequent=[f"{NAMES[a]} -> {NAMES[b]} ({mass[a,b]*100:.1f}%)" for a, b in fi])
    return out

desc = describe(m, F_tr)
for k, d in desc.items():
    print(f"\n=== cluster {k}  (soft share {d['share']:.3f}) ===")
    print("characteristic:"); [print("   ", s) for s in d["characteristic"]]
    print("frequent:");       [print("   ", s) for s in d["frequent"]]
json.dump(desc, open(f"{WORK}/results/markov_cluster_descriptions_K{K_FINAL}.json", "w"), indent=1)

## 3. Stability
* **Seed stability**: refit with different seeds (full restarts), pairwise ARI of the hard labels.
* **Bootstrap stability**: fit on random 80 % subsamples, label *all* learners, ARI against the reference clustering.
* **Train -> test**: do cluster proportions and per-cluster dropout rates carry over to the test learners (frozen train model)?

In [ ]:
seed_labels = [fit_best(F_tr, K_FINAL, n_init=1, seed=100 + s).predict(F_tr) for s in range(SEEDS_STAB)]
stab = C.seed_stability(seed_labels)
stab.update(C.bootstrap_stability(
    fit=lambda idx, seed: MarkovMixture(K_FINAL, V, ALPHA, seed=seed, cap=CAP).fit(F_tr[idx]),
    predict=lambda mm, rows: mm.predict(F_tr[rows]), n_rows=F_tr.shape[0], ref_labels=lab_tr, n_boot=N_BOOT))
stab.update(C.transfer_consistency(lab_tr, y_tr, lab_te, y_te, K_FINAL))
stab = {k: round(v, 4) for k, v in stab.items()}
print(pd.Series(stab).to_string())

## 4. Downstream dropout prediction (train CV, then the real test set)
Variants added to the baseline features:
* `hard` one-hot label, `soft` posteriors, `hard+soft`, `avgll` (per-cluster average log-likelihood, a "distance to centroid"-style soft output)
* `null_random`: a **random** one-hot label with the same K. It shows how much AUC change pure noise produces. Anything that does not clearly beat it is not signal.

The mixture is unsupervised (no dropout label), so fitting it on all train before CV is not label leakage. CV reports mean +- std over `CV_SPLITS x CV_REPEATS` folds
and the **paired** delta against the baseline on identical folds.

In [ ]:
def make_variants(model, K, F_a, seed=0):
    P = model.predict_proba(F_a); lab = P.argmax(1)
    rng = np.random.default_rng(seed)
    return {"hard": C.onehot(lab, K), "soft": P.astype("float32"),
            "hard+soft": np.hstack([C.onehot(lab, K), P]).astype("float32"),
            "avgll": model.avg_ll(F_a).astype("float32"),
            "null_random": C.onehot(rng.integers(0, K, F_a.shape[0]), K)}

cv_all, test_all = [], []
for K in (DOWNSTREAM_KS or [K_FINAL]):
    mk = models[K]
    var_tr, var_te = make_variants(mk, K, F_tr), make_variants(mk, K, F_te, seed=1)
    summ, wide = C.downstream_cv(A["base_train"], y_tr, var_tr, CV_SAMPLE, CV_SPLITS, CV_REPEATS)
    summ.insert(0, "K", K); cv_all.append(summ)
    print(f"\n=== train CV, K={K} ===");  display(summ)
    ft = C.final_test_eval(A["base_train"], y_tr, A["base_test"], y_te, var_tr, var_te)
    ft.insert(0, "K", K); test_all.append(ft)
    print(f"=== real TEST set, K={K} ==="); display(ft)
pd.concat(cv_all).to_csv(f"{WORK}/results/markov_downstream_cv.csv")
pd.concat(test_all).to_csv(f"{WORK}/results/markov_downstream_test.csv")

## 5. Save features and a one-line summary (for the cross-method comparison notebook)

In [ ]:
for s, FF in (("train", F_tr), ("test", F_te)):
    P = m.predict_proba(FF)
    np.save(f"{WORK}/results/feat_markov_K{K_FINAL}_{s}_soft.npy", P.astype("float32"))
    np.save(f"{WORK}/results/feat_markov_K{K_FINAL}_{s}_avgll.npy", m.avg_ll(FF).astype("float32"))
    pd.DataFrame(P, index=A[f"ids_{s}"], columns=[f"p{k}" for k in range(K_FINAL)]).assign(cluster=P.argmax(1)) \
      .to_csv(f"{WORK}/results/markov_labels_K{K_FINAL}_{s}.csv", index_label="enrollment_id")
np.savez(f"{WORK}/results/markov_model_K{K_FINAL}.npz", log_theta=m.log_theta, log_pi=m.log_pi)

summary = dict(method="markov_mixture", K=K_FINAL, cap=CAP)
summary.update({k: float(v) for k, v in grid.loc[K_FINAL].items()})
summary.update(stab)
json.dump(summary, open(f"{WORK}/results/summary_markov.json", "w"), indent=1)
print(json.dumps(summary, indent=1))